# TSN Defragmentation Visualization

This notebook uses a tiny deterministic TSN scenario so we can clearly see what the defragmentation actions do.

- Inline plots only (no popup windows)
- Uses existing `Schedule.plot()` from `schedule.py`
- Includes frame-by-frame snapshots and a GIF

In [ ]:
%matplotlib inline

import io
import os
import copy
import tempfile
import configparser

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from IPython.display import display, Image as IPyImage

from env import TSNDefragEnv
from tsn_scheduling_problem import TSNScheduling
from usecase_generator import Usecase
from scenario_tools import make_action_from_first_offsets

In [ ]:
def build_simple_usecase(config_file: str = "uc_config.cfg") -> Usecase:
    """
    Tiny deterministic topology:
    host-1 --> switch-1 --> host-2
    host-3 --> switch-1 --> host-2

    Streams s0 and s1 share egress `switch-1.Port2` (listener side),
    so defragmentation behavior is easy to inspect.
    """
    G = nx.DiGraph()

    # Nodes
    G.add_node("host-1", type="host", ports={"Port1": {"name": "Port1", "speed": 1000, "queueCount": 6}})
    G.add_node("host-2", type="host", ports={"Port1": {"name": "Port1", "speed": 1000, "queueCount": 6}})
    G.add_node("host-3", type="host", ports={"Port1": {"name": "Port1", "speed": 1000, "queueCount": 6}})
    G.add_node(
        "switch-1",
        type="switch",
        procDelay=40_000,
        ports={
            "Port1": {"name": "Port1", "speed": 1000, "queueCount": 6},
            "Port2": {"name": "Port2", "speed": 1000, "queueCount": 6},
            "Port3": {"name": "Port3", "speed": 1000, "queueCount": 6},
        },
    )

    # Edges with TSN port metadata
    # host-1 <-> switch-1
    G.add_edge("host-1", "switch-1", sourcePort="host-1.Port1", destPort="switch-1.Port1", sourceNode="host-1", destNode="switch-1", propDelay=10, linkBw=1000)
    G.add_edge("switch-1", "host-1", sourcePort="switch-1.Port1", destPort="host-1.Port1", sourceNode="switch-1", destNode="host-1", propDelay=10, linkBw=1000)

    # host-2 <-> switch-1
    G.add_edge("host-2", "switch-1", sourcePort="host-2.Port1", destPort="switch-1.Port2", sourceNode="host-2", destNode="switch-1", propDelay=10, linkBw=1000)
    G.add_edge("switch-1", "host-2", sourcePort="switch-1.Port2", destPort="host-2.Port1", sourceNode="switch-1", destNode="host-2", propDelay=10, linkBw=1000)

    # host-3 <-> switch-1
    G.add_edge("host-3", "switch-1", sourcePort="host-3.Port1", destPort="switch-1.Port3", sourceNode="host-3", destNode="switch-1", propDelay=10, linkBw=1000)
    G.add_edge("switch-1", "host-3", sourcePort="switch-1.Port3", destPort="host-3.Port1", sourceNode="switch-1", destNode="host-3", propDelay=10, linkBw=1000)

    # Large frames to make slots visually meaningful in a 2ms cycle
    streams = [
        {
            "name": "stream-0",
            "type": "isochronous",
            "interval": 2_000_000,
            "maxFramesPerInterval": 1,
            "maxFrameSize": 80_000,
            "maxLatency": 3_000_000,
            "tlkrPssblPlcmnt": ["host-1"],
            "lstnsrPssblPlcmnt": [["host-2"]],
            "pathLists": [["host-1.Port1", "switch-1.Port2", "host-2"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": 1_900_000,
            "jitter": 0,
        },
        {
            "name": "stream-1",
            "type": "isochronous",
            "interval": 2_000_000,
            "maxFramesPerInterval": 1,
            "maxFrameSize": 70_000,
            "maxLatency": 3_000_000,
            "tlkrPssblPlcmnt": ["host-3"],
            "lstnsrPssblPlcmnt": [["host-2"]],
            "pathLists": [["host-3.Port1", "switch-1.Port2", "host-2"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": 1_900_000,
            "jitter": 0,
        },
        {
            "name": "stream-2",
            "type": "isochronous",
            "interval": 2_000_000,
            "maxFramesPerInterval": 1,
            "maxFrameSize": 50_000,
            "maxLatency": 3_000_000,
            "tlkrPssblPlcmnt": ["host-1"],
            "lstnsrPssblPlcmnt": [["host-3"]],
            "pathLists": [["host-1.Port1", "switch-1.Port3", "host-3"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": 1_900_000,
            "jitter": 0,
        },
    ]

    uc = Usecase(configFile=config_file, topology=G, streams=streams)
    uc.index = 0

    # Force deterministic stream config assumptions used by TSNScheduling metadata.
    uc.config.set("streams_config", "numplacestalker", "1")
    uc.config.set("streams_config", "numplaceslistener", "1")
    uc.config.set("streams_config", "routing", "{'fixPath': True, 'shortestPath': True}")
    return uc

In [ ]:
problem = TSNScheduling(problem_definition="uc_config.cfg")
usecase = build_simple_usecase()

env = TSNDefragEnv(
    problem=problem,
    usecase=usecase,
    gap_threshold_ratio=0.08,
    move_penalty=0.01,
    fragmentation_metric="hybrid",
)
obs, _ = env.reset()

# Build an intentionally fragmented initial state (known bad layout).
# These are first-hop offsets in ns.
fragmented_offsets = [0.0, 760_000.0, 1_520_000.0]
for i, off in enumerate(fragmented_offsets):
    env._place_stream(i, off)

print("n_streams:", env.n_streams, "n_ports:", env.n_ports, "cycle_ns:", env.cycle_ns)
print("Initial frag:", env._fragmentation_value(), "all_gaps:", env._count_gaps(), "useless_gaps:", env._count_useless_gaps())

In [ ]:
sched_before = env.get_combined_schedule()
fig = sched_before.plot(tick=10_000, label=True)
fig.suptitle("Before defragmentation", fontsize=14)
plt.show()

In [ ]:
def eval_offsets(env, first_offsets):
    """Evaluate one candidate offset assignment on a copy of env (no side effects)."""
    env_copy = copy.deepcopy(env)
    action = make_action_from_first_offsets(env_copy, first_offsets)
    _, reward, _, _, info = env_copy.step(action)
    return reward, info


def greedy_target_offsets(env):
    """
    Very simple deterministic policy:
    sort streams by first-hop duration descending and pack them from offset 0.
    This gives us a known expected behavior for this toy scenario.
    """
    order = []
    for s_idx, sinfo in enumerate(env.stream_info):
        p0 = sinfo["path"][0]
        d0 = sinfo["port_dur_ns"][p0]
        order.append((s_idx, d0))
    order.sort(key=lambda x: x[1], reverse=True)

    target = [0.0] * env.n_streams
    cursor = 0.0
    for s_idx, d0 in order:
        target[s_idx] = cursor
        cursor += d0
    return target


target_offsets = greedy_target_offsets(env)
reward_preview, info_preview = eval_offsets(env, target_offsets)
print("Target offsets:", [int(x) for x in target_offsets])
print("Preview reward:", reward_preview)
print("Preview delta_frag:", info_preview["delta_frag"])

In [ ]:
# Frame-by-frame transition from current offsets to greedy target.
cur_offsets = []
for s_idx, sinfo in enumerate(env.stream_info):
    p0 = sinfo["path"][0]
    cur_offsets.append(env.stream_offsets[s_idx][p0])

n_steps = 6
frame_infos = []
frame_png_paths = []
tmp_dir = tempfile.mkdtemp(prefix="defrag_frames_")

for step in range(n_steps + 1):
    alpha = step / n_steps
    step_offsets = [
        (1 - alpha) * cur_offsets[i] + alpha * target_offsets[i]
        for i in range(env.n_streams)
    ]

    action = make_action_from_first_offsets(env, step_offsets)
    _, reward, _, _, info = env.step(action)

    sched = env.get_combined_schedule()
    fig = sched.plot(tick=10_000, label=True)
    fig.suptitle(f"Frame {step}/{n_steps} | reward={reward:.4f} | frag={info['new_fragmentation']:.4f}")

    frame_path = os.path.join(tmp_dir, f"frame_{step:02d}.png")
    fig.savefig(frame_path, dpi=110, bbox_inches="tight")
    plt.close(fig)

    frame_png_paths.append(frame_path)
    frame_infos.append((step, reward, info["new_fragmentation"], info["all_gaps"], info["useless_gaps"]))

print("Saved frames in:", tmp_dir)
frame_infos

In [ ]:
# Show first and last frame inline
display(IPyImage(filename=frame_png_paths[0]))
display(IPyImage(filename=frame_png_paths[-1]))

In [ ]:
# Build and show GIF inline
gif_path = os.path.join(tmp_dir, "defrag_episode.gif")
images = [PILImage.open(p).convert("P") for p in frame_png_paths]
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=850,
    loop=0,
)

print("GIF:", gif_path)
display(IPyImage(filename=gif_path))

In [ ]:
# Metric traces for the episode
steps = [x[0] for x in frame_infos]
rewards = [x[1] for x in frame_infos]
frags = [x[2] for x in frame_infos]
all_gaps = [x[3] for x in frame_infos]
useless_gaps = [x[4] for x in frame_infos]

fig, axs = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axs[0].plot(steps, rewards, marker="o")
axs[0].set_ylabel("Reward")
axs[0].grid(True, alpha=0.3)

axs[1].plot(steps, frags, marker="o")
axs[1].set_ylabel("Fragmentation")
axs[1].grid(True, alpha=0.3)

axs[2].plot(steps, all_gaps, marker="o", label="all_gaps")
axs[2].plot(steps, useless_gaps, marker="o", label="useless_gaps")
axs[2].set_ylabel("Gap counts")
axs[2].set_xlabel("Frame")
axs[2].legend()
axs[2].grid(True, alpha=0.3)

fig.suptitle("Episode metrics (frame-by-frame)")
plt.tight_layout()
plt.show()